# Day 18: Magic Methods — Making Your Classes Pythonic
## __str__, __repr__, __len__, __eq__, and Operator Overloading

---

## What Are Magic (Dunder) Methods?

Magic methods (also called **dunder methods** — short for "double underscore") are special methods surrounded by **double underscores** (`__method__`). Python calls them **automatically** in specific situations — you don't call them directly.

### The Big Idea

```python
# Without magic methods — you call methods explicitly:
obj.to_string()
obj.get_length()
obj.equals(other)
obj.add(other)

# With magic methods — Python calls them FOR you:
print(obj)        # → Python calls obj.__str__()
len(obj)          # → Python calls obj.__len__()
obj == other      # → Python calls obj.__eq__(other)
obj + other       # → Python calls obj.__add__(other)
```



### What We'll Cover Today

| Method | Triggered By | What It Does |
|--------|-------------|--------------|
| `__str__` | `print(obj)`, `str(obj)` | User-friendly string |
| `__repr__` | `repr(obj)`, lists, debugger | Developer-friendly string |
| `__len__` | `len(obj)` | Number of items |
| `__contains__` | `item in obj` | Membership check |
| `__eq__` | `obj1 == obj2` | Equality comparison |
| `__lt__`, `__gt__` | `<`, `>`, `sorted()` | Ordering / sorting |
| `__add__`, `__mul__` | `+`, `*` | Operator overloading |

By the end of this notebook, you'll know how to make your own classes feel like they're part of Python itself!

## 1. __str__ vs __repr__ — Two Ways to Represent an Object

### The Difference

- **`__str__`** answers: **"What is this object?"** → for end users, `print()`, and logs
- **`__repr__`** answers: **"How was this object created?"** → for developers, debugging, and error messages

Think of it like this:
- `__str__` is like a **name tag** — friendly and readable
- `__repr__` is like a **blueprint** — shows exactly how to recreate the object

### The Golden Rule of `__repr__`

> **`__repr__` should return a string that looks like valid Python code to recreate the object.**

```python
# Good __repr__:
Point(x=3, y=5)     # You could copy-paste this to recreate the object!

# Bad __repr__:
"Point at 3, 5"     # Ambiguous — can't recreate from this
```

### When Each Is Called

| Context | Uses |
|---------|------|
| `print(obj)`, `str(obj)`, f`"{obj}"` | `__str__` |
| `repr(obj)`, f`"{obj!r}"`, displaying in a list, debugger | `__repr__` |

### What If You Only Define One?

- **Only `__repr__`** → `__str__` falls back to it. This is safe!
- **Only `__str__`** → `__repr__` shows `<Point object at 0x7f...>` — ugly and useless for debugging.

> **Best practice:** Always define at least `__repr__`. Defining both is ideal.

In [12]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # Called by print() — nice, readable format
        return f"Point({self.x}, {self.y})"

    def __repr__(self):
        # Called in lists, debugging — shows how to recreate the object
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 5)
print(p)                     # Uses __str__: Point(3, 5)
print(repr(p))               # Uses __repr__: Point(x=3, y=5)

# In a list, Python uses __repr__
points = [Point(1, 2), Point(3, 4)]
print(points)                # [Point(x=1, y=2), Point(x=3, y=4)]


Point(3, 5)
Point(x=3, y=5)
[Point(x=1, y=2), Point(x=3, y=4)]


## 📌 Quick Rule: `__str__()` vs `__repr__()`

- `print(object)` → Calls **`__str__()`**
- `repr(object)` → Calls **`__repr__()`**
- Objects inside containers (`list`, `tuple`, `set`, `dict`) → Python uses **`__repr__()`** for each object.

### Example

```python
p = Point(3, 5)

print(p)        # __str__()
print(repr(p))  # __repr__()

points = [Point(1, 2), Point(3, 4)]
print(points)   # __repr__() for each object
```

### Output

```text
Point(3, 5)
Point(x=3, y=5)
[Point(x=1, y=2), Point(x=3, y=4)]
```

> **Memory Trick:**
>
> - `__str__()` → Human-friendly output 👤
> - `__repr__()` → Developer/debug-friendly output 👨‍💻
> - Inside containers (`[]`, `()`, `{}`), Python always uses `__repr__()`.

## 2. __len__ and __contains__ — Make Your Class Work Like a Collection

### The Idea

Python has built-in functions like `len()` and the `in` keyword. By implementing `__len__` and `__contains__`, your objects can work with them too — just like lists and dictionaries!

| Method | Built-in / Syntax | What It Does |
|--------|------------------|--------------|
| `__len__(self)` | `len(obj)` | Return the number of items |
| `__contains__(self, item)` | `item in obj` | Check if an item exists in the object |

### A Simple Mental Model

```python
# When you write:
len(my_playlist)

# Python does:
my_playlist.__len__()

# When you write:
"song" in my_playlist

# Python does:
my_playlist.__contains__("song")
```

That's it! You define what `len()` and `in` mean for your class, and Python handles the rest.

In [13]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add_song(self, song):
        self.songs.append(song)

    def __len__(self):
        # Called by len(playlist)
        return len(self.songs)

    def __contains__(self, song):
        # Called by 'song in playlist'
        return song in self.songs

    def __str__(self):
        return f"Playlist '{self.name}' with {len(self)} songs"

my_playlist = Playlist("Chill Vibes")
my_playlist.add_song("Yesterday")
my_playlist.add_song("Imagine")
my_playlist.add_song("Bohemian Rhapsody")

print(my_playlist)                       # Playlist 'Chill Vibes' with 3 songs
print(f"Number of songs: {len(my_playlist)}")   # 3
print(f"Is 'Imagine' in playlist? {'Imagine' in my_playlist}")  # True
print(f"Is 'Despacito' in playlist? {'Despacito' in my_playlist}")  # False


Playlist 'Chill Vibes' with 3 songs
Number of songs: 3
Is 'Imagine' in playlist? True
Is 'Despacito' in playlist? False


## 3. __eq__, __lt__, __gt__ — Comparison Operators

### The Problem

By default, Python doesn't know how to compare YOUR custom objects:

```python
student1 = Student("Rahul", 85)
student2 = Student("Priya", 92)

print(student1 == student2)    # False (different objects, even if marks are same!)
print(student1 > student2)     # ERROR! Python doesn't know how to compare
```

### The Solution: Comparison Magic Methods

| Method | Operator | Meaning |
|--------|----------|---------|
| `__eq__(self, other)` | `==` | Are they equal? |
| `__lt__(self, other)` | `<` | Is self less than other? |
| `__gt__(self, other)` | `>` | Is self greater than other? |

Once you define these, `sorted()`, `max()`, and `min()` all work automatically!

### Important: Always Check the Type

```python
def __eq__(self, other):
    if not isinstance(other, Student):    # ← Guard against wrong types!
        return False
    return self.marks == other.marks
```

Without this check, comparing a `Student` to an `int` could crash your program. Always verify `other` is the right type before comparing.

In [2]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def __eq__(self, other):
        # Called by ==
        if not isinstance(other, Student):
            return False
        return self.marks == other.marks

    def __lt__(self, other):
        # Called by <  (also enables sorting with sorted())
        return self.marks < other.marks

    def __gt__(self, other):
        # Called by >
        return self.marks > other.marks

    def __str__(self):
        return f"{self.name} ({self.marks}%)"

students = [
    Student("Rahul", 85),
    Student("Priya", 92),
    Student("Amit", 78),
]


# Compare students
print(f"Is Rahul == Priya? {students[0] == students[1]}")  # False (different marks)
print(f"Is Rahul > Amit? {students[0] > students[2]}")     # True (85 > 78)

# Sort students (uses __lt__)
sorted_students = sorted(students)
print("\nSorted by marks:")
for s in sorted_students:
    print(f"  {s}")

# Find top student (uses __gt__)
top = max(students)
print(f"\nTop student: {top}")


Is Rahul == Priya? False
Is Rahul > Amit? True

Sorted by marks:
  Amit (78%)
  Rahul (85%)
  Priya (92%)

Top student: Priya (92%)


## 4. __add__, __mul__, __sub__ — Operator Overloading

### The Idea

What if you could add two objects with `+` just like you add two numbers? That's operator overloading!

| Method | Operator | Example |
|--------|----------|---------|
| `__add__(self, other)` | `+` | `v1 + v2` |
| `__sub__(self, other)` | `-` | `v1 - v2` |
| `__mul__(self, other)` | `*` | `v1 * 3` |

### How It Works

```python
# When you write:
v1 + v2

# Python translates it to:
v1.__add__(v2)

# Whatever __add__ returns becomes the result of v1 + v2!
```

### A Real-World Example: 2D Vectors

Think of a vector as an arrow pointing from (0,0) to (x,y). Adding two vectors means adding their x components and y components separately:

```
(3, 4) + (1, 2) = (3+1, 4+2) = (4, 6)
```

By defining `__add__`, you can write `Vector(3,4) + Vector(1,2)` and get `Vector(4,6)` — just like math class!

In [15]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        # Called by v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        # Called by v * 3 (scalar multiplication)
        return Vector(self.x * scalar, self.y * scalar)

    def __sub__(self, other):
        # Called by v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __str__(self):
        return f"Vector({self.x}, {self.y})"

    def magnitude(self):
        """Returns the length (magnitude) of the vector."""
        return (self.x**2 + self.y**2) ** 0.5

v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"v1 = {v1}, magnitude = {v1.magnitude():.1f}")
print(f"v2 = {v2}")
print(f"v1 + v2 = {v1 + v2}")    # Uses __add__
print(f"v1 - v2 = {v1 - v2}")    # Uses __sub__
print(f"v1 * 3 = {v1 * 3}")      # Uses __mul__


v1 = Vector(3, 4), magnitude = 5.0
v2 = Vector(1, 2)
v1 + v2 = Vector(4, 6)
v1 - v2 = Vector(2, 2)
v1 * 3 = Vector(9, 12)


---

# Summary: Magic Methods Cheat Sheet

| You Want... | Implement This | Then You Can... |
|-------------|---------------|------------------|
| Pretty printing | `__str__` | `print(obj)`, `str(obj)` |
| Debug representation | `__repr__` | `repr(obj)`, display in lists |
| Length of object | `__len__` | `len(obj)` |
| Membership test | `__contains__` | `x in obj` |
| Equality comparison | `__eq__` | `obj1 == obj2` |
| Sorting / ordering | `__lt__`, `__gt__` | `sorted()`, `max()`, `min()`, `<`, `>` |
| Addition | `__add__` | `obj1 + obj2` |
| Subtraction | `__sub__` | `obj1 - obj2` |
| Multiplication | `__mul__` | `obj * scalar` |

---

# Exercises

Practice makes perfect! Try these exercises to solidify your understanding.

### Exercise 1: Book (Easy)
Create a `Book` class with attributes `title`, `author`, and `isbn`. Implement:
- `__eq__` — two books are equal if they have the **same ISBN**
- `__str__` — returns `"Title by Author"` (e.g., `"1984 by George Orwell"`)
- `__repr__` — returns `Book(title='...', author='...', isbn='...')`

### Exercise 2: ShoppingCart (Medium)
Create a `ShoppingCart` class that stores items in a list. Implement:
- `__len__` — returns number of items in the cart
- `__contains__` — checks if an item is in the cart
- `__add__` — merges two carts into a new cart (returns a new `ShoppingCart`)

### Exercise 3: Fraction (Medium)
Create a `Fraction` class with `numerator` and `denominator`. Implement:
- `__add__` — correctly adds two fractions: `a/b + c/d = (a*d + b*c) / (b*d)`
- `__mul__` — multiplies two fractions: `a/b * c/d = (a*c) / (b*d)`
- `__eq__` — checks if two fractions represent the same value
- `__str__` — displays as `"a/b"`

---

## Key Takeaways

1. **Magic methods make your classes feel like built-in Python types** — that's the whole point!

2. **Always define `__repr__`** — it makes debugging so much easier. If you only define one, make it `__repr__`.

3. **`__str__` is for users (readable), `__repr__` is for developers (precise)** — know the difference.

4. **Defining `__eq__` + `__lt__` unlocks `sorted()`, `max()`, `min()`** — powerful with very little code.

5. **Always check the type with `isinstance()` in `__eq__`** — prevents crashes when comparing different types.

6. **Operator overloading makes math code intuitive** — `v1 + v2` reads exactly like the math you'd write on paper.

---

## What's Next?

In Day 19, we'll explore **OOP Inheritance** — how classes can inherit behavior from parent classes, override methods, and use `super()` to build class hierarchies.